# EVA Phase 3 — Train Learning NER

Trains **learning** NER on `learning_ner.json` with:
- Correct BIO subword alignment (Phase 3 fix)
- Entity-level F1 evaluation (seqeval)
- Base model: `bert-base-cased`

**Deploy output to:** `Eva_AI_Server/models/learning_ner/`

See `COLAB_GUIDE.md` for upload instructions.

## 1. Install seqeval

In [ ]:
!pip install -q seqeval scikit-learn

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- EDIT THIS PATH ---
DATA_ROOT = "/content/drive/MyDrive/NER_Improvement"

# Or: DATA_ROOT = "/content/eva_ner"

## 3. Config

In [ ]:
from pathlib import Path

ROOT = Path(DATA_ROOT)
DATASET_PATH = ROOT / "datasets/learning/raw/learning_ner.json"
LABELS_PATH = ROOT / "config/labels.json"
LABEL_KEY = "learning_ner"

OUTPUT_DIR = Path("/content/learning_ner_output")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
FINAL_DIR = OUTPUT_DIR / "final"

BASE_MODEL = "bert-base-cased"
MAX_LEN = 128
EPOCHS = 10
BATCH_SIZE = 16
LR = 3e-5
SEED = 42
VAL_RATIO = 0.1

assert DATASET_PATH.exists(), f"Dataset not found: {DATASET_PATH}"
assert LABELS_PATH.exists(), f"Labels not found: {LABELS_PATH}"
print("Dataset:", DATASET_PATH)
print("Labels:", LABELS_PATH)

## 4. Training utilities (BIO alignment fix)

In [ ]:
import json
import random
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Sequence

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)


@dataclass
class LabelMaps:
    labels: List[str]
    label2id: Dict[str, int]
    id2label: Dict[int, str]


def load_label_maps(path, label_key):
    with open(path, encoding="utf-8") as f:
        cfg = json.load(f)
    block = cfg[label_key]
    label2id = {k: int(v) for k, v in block["label2id"].items()}
    id2label = {int(k): v for k, v in block["id2label"].items()}
    return LabelMaps(block["labels"], label2id, id2label)


def load_ner_dataset(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    out = []
    for ex in data:
        tokens, labels = ex.get("tokens", []), ex.get("labels", [])
        if tokens and len(tokens) == len(labels):
            out.append({"tokens": tokens, "labels": labels, "intent": ex.get("intent", "")})
    return out


def align_labels_with_tokens(word_labels, word_ids, label2id):
    """Phase 3 fix: continuation subwords of entities get I- tags, not -100."""
    aligned, prev = [], None
    for word_idx in word_ids:
        if word_idx is None:
            aligned.append(-100)
            continue
        wl = word_labels[word_idx]
        if word_idx != prev:
            aligned.append(label2id.get(wl, label2id["O"]))
        else:
            if wl == "O":
                aligned.append(-100)
            elif wl.startswith("B-"):
                aligned.append(label2id.get("I-" + wl[2:], -100))
            else:
                aligned.append(label2id.get(wl, -100))
        prev = word_idx
    return aligned


def tokenize_examples(examples, tokenizer, label_maps, max_len):
    enc = {"input_ids": [], "attention_mask": [], "labels": []}
    for ex in examples:
        e = tokenizer(ex["tokens"], is_split_into_words=True, truncation=True, max_length=max_len)
        enc["input_ids"].append(e["input_ids"])
        enc["attention_mask"].append(e["attention_mask"])
        enc["labels"].append(align_labels_with_tokens(ex["labels"], e.word_ids(), label_maps.label2id))
    return enc


class NERDataset(torch.utils.data.Dataset):
    def __init__(self, enc):
        self.enc = enc
    def __len__(self):
        return len(self.enc["input_ids"])
    def __getitem__(self, i):
        return {
            "input_ids": torch.tensor(self.enc["input_ids"][i]),
            "attention_mask": torch.tensor(self.enc["attention_mask"][i]),
            "labels": torch.tensor(self.enc["labels"][i]),
        }


def compute_metrics_builder(id2label):
    def compute_metrics(eval_pred):
        preds, labels = eval_pred
        preds = np.argmax(preds, axis=2)
        true, pred = [], []
        for pr, lb in zip(preds, labels):
            gt, pt = [], []
            for p, l in zip(pr, lb):
                if l == -100: continue
                gt.append(id2label[int(l)])
                pt.append(id2label[int(p)])
            true.append(gt); pred.append(pt)
        return {"precision": precision_score(true, pred), "recall": recall_score(true, pred), "f1": f1_score(true, pred)}
    return compute_metrics


demo = align_labels_with_tokens(
    ["O", "B-topic", "O"],
    [None, 0, 1, 1, 2, None],
    {"O": 0, "B-topic": 1, "I-topic": 2},
)
assert demo[3] == 2, "Alignment fix failed!"
print("BIO alignment fix OK ✓")

## 5. Load data & tokenize

In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

label_maps = load_label_maps(LABELS_PATH, LABEL_KEY)
raw = load_ner_dataset(DATASET_PATH)
print(f"Loaded {len(raw)} examples")
print(f"Labels ({len(label_maps.labels)}):", label_maps.labels)

stratify = [x["intent"] for x in raw] if len(set(x["intent"] for x in raw)) > 1 else None
train_raw, val_raw = train_test_split(raw, test_size=VAL_RATIO, random_state=SEED, stratify=stratify)
print(f"Train: {len(train_raw)}, Val: {len(val_raw)}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
train_enc = tokenize_examples(train_raw, tokenizer, label_maps, MAX_LEN)
val_enc = tokenize_examples(val_raw, tokenizer, label_maps, MAX_LEN)
train_ds = NERDataset(train_enc)
val_ds = NERDataset(val_enc)
print("Tokenized ✓")

## 6. Train

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(label_maps.labels),
    id2label=label_maps.id2label,
    label2id=label_maps.label2id,
)

args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics_builder(label_maps.id2label),
)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
trainer.train()

## 7. Entity F1 report

In [ ]:
preds = trainer.predict(val_ds)
pred_ids = np.argmax(preds.predictions, axis=2)

true_labels, pred_labels = [], []
for pr, lb in zip(pred_ids, preds.label_ids):
    gt, pt = [], []
    for p, l in zip(pr, lb):
        if l == -100: continue
        gt.append(label_maps.id2label[int(l)])
        pt.append(label_maps.id2label[int(p)])
    true_labels.append(gt); pred_labels.append(pt)

report = classification_report(true_labels, pred_labels, digits=4)
overall_f1 = f1_score(true_labels, pred_labels)
print(report)
print(f"\nOverall entity F1: {overall_f1:.4f}")

f1_report = {
    "model": "learning_ner",
    "overall_f1": overall_f1,
    "classification_report": report,
    "epochs": EPOCHS,
    "train_size": len(train_raw),
    "val_size": len(val_raw),
}
with open(OUTPUT_DIR / "baseline_f1_learning.json", "w") as f:
    json.dump(f1_report, f, indent=2)
print("F1 report saved.")

## 8. Save model & download

Copy `final/` contents to `Eva_AI_Server/models/learning_ner/`

In [ ]:
FINAL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))
print(f"Model saved to {FINAL_DIR}")

!cd /content && zip -r learning_ner_model.zip learning_ner_output/final learning_ner_output/baseline_f1_learning.json
from google.colab import files
files.download("/content/learning_ner_model.zip")

## 9. Quick inference test

In [ ]:
test_sentences = [
    "learn Python for 2 weeks",
    "Python basics start from Monday",
    "delete my JavaScript roadmap",
    "show Python progress",
]

model.eval()
for text in test_sentences:
    tokens = text.split()
    enc = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
    pred_ids = torch.argmax(logits, dim=-1)[0].tolist()
    wids = tokenizer(tokens, is_split_into_words=True).word_ids()
    tags = {}
    for idx, wid in enumerate(wids):
        if wid is not None and wid not in tags:
            tags[wid] = label_maps.id2label[pred_ids[idx]]
    tagged = [(t, tags.get(i, "O")) for i, t in enumerate(tokens)]
    print(f"\n{text}")
    print(tagged)